# 04 · Train, export, run: a direction detector, learned

Notebook 02's task, larger: an 8 × 8 event camera sees a bar moving right, left, down or up, at three speeds and with noise. A network learns it in snnTorch, is converted with the rule of notebook 03 and runs in py-spinnaker2 on Brian2 and, for cluster participants, on the chip (section 7).

**Try** marks exercises, **Discuss** questions we take up together, ▶ optional background.

## 0. Setup

In [ ]:
# --- libraries ---
import contextlib, io  # to silence the Brian2 progress printout
import numpy as np  # arrays and numbers
import matplotlib.pyplot as plt  # figures
from matplotlib.animation import FuncAnimation, PillowWriter  # animations, saved as GIFs
from IPython.display import Image, display  # shows the GIFs in the notebook
import torch  # PyTorch: tensors and training
import torch.nn as nn  # layers (nn.Linear holds the weights)
import torch.nn.functional as F  # the loss function
import snntorch as snn  # spiking layers for PyTorch (snn.Leaky is the LIF neuron)
from snntorch import surrogate, utils  # surrogate gradients for training; utils.reset clears the potentials
from snntorch.export_nir import export_to_nir  # snnTorch -> NIR
from spinnaker2 import brian2_sim, s2_nir  # py-spinnaker2: the Brian2 backend and the NIR import

# --- backend: where the networks run ---
brian2 = brian2_sim.Brian2Backend()  # the simulator on this computer; section 7 runs the network on the chip
torch.manual_seed(0)  # the same training result on every run

# --- figure style ---
plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.color": "0.88", "grid.linewidth": 0.6})  # light grey grid in all figures

### Helpers for figures and animations

Functions that only draw; nothing about the network. Run the cell; its code is collapsed (click it to expand).

In [ ]:
def animate(examples, titles, filename, trail=True):
    """Save and show a looping GIF of examples (each an array: time steps x rows x columns), one panel per example."""
    steps, rows, cols = examples[0].shape
    fade = 0.6 if trail else 0.0  # older events fade (only for the eye)
    fig, axes = plt.subplots(1, len(examples), figsize=(2.5 * len(examples), 3.3), squeeze=False)
    images, traces = [], [np.zeros((rows, cols)) for _ in examples]
    for ax, title in zip(axes[0], titles):
        images.append(ax.imshow(np.zeros((rows, cols)), cmap="gray_r", vmin=0, vmax=1))
        ax.set_title(title, fontsize=10)
        ax.set(xticks=[], yticks=[])
        ax.grid(False)
    step_text = fig.suptitle("")

    def update(t):
        for i, ex in enumerate(examples):
            traces[i] = fade * traces[i] + ex[t]  # new events at full strength, older ones fade (or vanish with fade 0)
            images[i].set_data(np.clip(traces[i], 0, 1))
        step_text.set_text(f"time step {t}")
        return images

    plt.tight_layout(rect=(0, 0, 1, 0.88))  # room for the step counter above the titles
    FuncAnimation(fig, update, frames=steps, interval=200).save(filename, writer=PillowWriter(fps=5))
    plt.close(fig)
    display(Image(filename=filename))

## 1. The task: a bar moving across an 8 × 8 event camera

An event camera has no frames: each pixel fires an event when the brightness at that pixel changes. A bar of three pixels moving across the sensor therefore makes a short trail of events. Each example is 30 time steps long; the bar moves left, right, up or down, one pixel every 1, 2 or 3 steps, starting at a random time and position. To make it realistic, 10 % of the bar's events are lost, and every pixel fires now and then on its own (background noise), about as many noise events as bar events.

The generator below makes the examples; the animation shows one per direction. The fading is only for the eye: the network gets only the events of each step, and its memory of earlier events lies in the membrane potentials of its neurons. `show_trail = False` switches the fading off.

In [ ]:
# --- change these ---
noise = 0.01  # probability that a pixel fires on its own in a time step
dropout = 0.1  # share of the bar's events that get lost
show_trail = True  # animations: True: older events fade out (for the eye only); False: only the events of each step, as the network gets them

# --- the generator ---
SIZE, STEPS = 8, 30
DIRECTIONS = ["right", "left", "down", "up"]  # label 0, 1, 2, 3
rng = np.random.default_rng(0)


def make_example(direction, rng):
    """One example: events (time steps, rows, columns) with 1 where a pixel fires, and the speed (steps per pixel)."""
    events = np.zeros((STEPS, SIZE, SIZE), dtype=np.float32)
    every = rng.integers(1, 4)  # the bar moves one pixel every 1, 2 or 3 steps
    start = rng.integers(0, 6)  # time step at which it appears
    across = rng.integers(0, SIZE - 2)  # position of the bar across the direction of motion
    for k, t in enumerate(range(start, STEPS, every)):  # k = how far the bar has moved
        if k >= SIZE:
            break
        along = k if direction in ("right", "down") else SIZE - 1 - k
        for i in range(3):  # the three pixels of the bar
            row, col = (across + i, along) if direction in ("right", "left") else (along, across + i)
            if rng.random() >= dropout:
                events[t, row, col] = 1
    events[rng.random(events.shape) < noise] = 1  # background noise
    return events, every


def make_set(n, rng):
    """n random examples -> inputs (time steps, n, 64 pixels), labels (n,) and speeds (n,), in steps per pixel."""
    labels = rng.integers(0, 4, n)
    examples = [make_example(DIRECTIONS[c], rng) for c in labels]
    events = np.stack([ev for ev, _ in examples], axis=1)
    speeds = np.array([every for _, every in examples])
    return torch.tensor(events.reshape(STEPS, n, SIZE * SIZE)), torch.tensor(labels), speeds


# --- one example per direction ---
one_per_direction = [make_example(d, rng) for d in DIRECTIONS]
titles = [f"{d}, 1 px / {every} step{'s' if every > 1 else ''}" for d, (_, every) in zip(DIRECTIONS, one_per_direction)]
animate([ev for ev, _ in one_per_direction], titles, "direction_examples.gif", trail=show_trail)

<details><summary>Why a slow bar seems to disappear between its moves</summary>

An event camera reports changes, not images: a pixel fires only when the brightness at that pixel changes. When the bar moves onto a pixel, the pixel fires; while the bar stays where it is, nothing changes and no pixel fires. A bar that moves one pixel every 3 steps is therefore visible in one step and invisible in the next two (switch the trail off with `show_trail = False` to see it). A real sensor also fires when the bar leaves a pixel, as a separate OFF event (darker instead of brighter); this generator only produces the events at the front edge of the bar.

</details>

For training, the network needs many examples with the right answer (the label). The **training set** is what it learns from; the **test set** is only used to check whether it learned the task or just memorised its examples.

In [ ]:
x_train, y_train, speed_train = make_set(2000, rng)  # inputs, labels, speeds
x_test, y_test, speed_test = make_set(200, np.random.default_rng(1))  # own random generator: the test set stays the same when the training set changes
print(f"training set: {tuple(x_train.shape)} (time steps, examples, pixels); test set: {x_test.shape[1]} examples")
print(f"events per example: {x_train.sum(dim=(0, 2)).mean():.0f} on average, about {noise * STEPS * SIZE * SIZE:.0f} of them noise")

## 2. The network in snnTorch

64 input pixels, a hidden layer of LIF neurons and four outputs, one per direction; the answer is the output that fires most often. The layers are those of notebook 03, without delays: the network has to find its own way to tell the direction from the events.

In [ ]:
# --- change these ---
n_hidden = 32  # neurons in the hidden layer
beta = 0.8  # share of the potential kept per step, below 1 (1.0, no leak at all, cannot be converted to py-spinnaker2)
epochs = 20  # passes through the training set

# --- build ---
torch.manual_seed(0)  # the same starting weights on every run, so that the Try results stay comparable
spike_grad = surrogate.fast_sigmoid()  # smooth stand-in for the step function, used only for training (section 3)
net = nn.Sequential(
    nn.Linear(SIZE * SIZE, n_hidden, bias=False),  # weights pixels -> hidden
    snn.Leaky(beta=beta, init_hidden=True, reset_mechanism="zero", spike_grad=spike_grad),  # LIF neurons, reset to 0 after a spike
    nn.Linear(n_hidden, 4, bias=False),  # weights hidden -> outputs
    snn.Leaky(beta=beta, init_hidden=True, reset_mechanism="zero", spike_grad=spike_grad, output=True),  # 4 LIF outputs, one per direction
)


# --- run it and read the answer ---
def spike_counts(x):
    """Run the network step by step over inputs (time steps, examples, pixels); return the spike count of each output."""
    utils.reset(net)  # start every run from potential 0
    out = [net(x_t)[0] for x_t in x]  # one time step of all examples
    return torch.stack(out).sum(dim=0)  # (examples, 4)


def accuracy(x, y):
    """Share of examples whose right output neuron fires most often (no spikes at all counts as wrong)."""
    with torch.no_grad():
        counts = spike_counts(x)
    return ((counts.argmax(dim=1) == y) & (counts.max(dim=1).values > 0)).float().mean().item()


print(net)

## 3. Train it

As in notebook 03, the loss is the cross-entropy of the four spike counts. The training set is shown in batches of 50 examples; one pass through all of them is an epoch. 20 epochs take about half a minute.

<details><summary>Surrogate gradients in two sentences</summary>

In the forward pass the neuron fires with the hard threshold, exactly as on the chip. In the backward pass the derivative of the step, which is zero almost everywhere, is replaced by the derivative of a smooth function (here a fast sigmoid), so that backpropagation through time can tell each weight in which direction to move. More in the workshop note [Spiking Neurons and Networks, section 8](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning).

</details>

In [ ]:
# --- training ---
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
history = []  # (epoch, training accuracy, test accuracy)

for epoch in range(1, epochs + 1):
    order = torch.randperm(x_train.shape[1])
    for i in range(0, len(order), 50):  # batches of 50 examples
        batch = order[i:i + 50]
        loss = F.cross_entropy(spike_counts(x_train[:, batch]), y_train[batch])  # like a classifier
        optimizer.zero_grad()
        loss.backward()  # backpropagation through time, with the surrogate gradient
        optimizer.step()
    history.append((epoch, accuracy(x_train, y_train), accuracy(x_test, y_test)))
    print(f"epoch {epoch:2d}: training {history[-1][1]:.0%}, test {history[-1][2]:.0%}")

# --- learning curve ---
epoch_ids, train_acc, test_acc = zip(*history)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epoch_ids, train_acc, "o-", label="training set")
ax.plot(epoch_ids, test_acc, "s-", label="test set")
ax.axhline(0.25, ls="--", c="0.5")
ax.text(epochs, 0.27, "guessing (25 %)", ha="right", color="0.4")
ax.set(xlabel="epoch", ylabel="accuracy", ylim=(0, 1.02), xticks=range(0, epochs + 1, 5))
ax.legend(loc="lower right")
plt.show()

## 4. Test it: where does it go wrong?

The test set contains examples the network has never seen. Below: the accuracy per direction and per speed, and the examples it got wrong, as an animation with the right answer and the network's answer.

In [ ]:
# --- accuracy per direction and per speed ---
with torch.no_grad():
    test_counts = spike_counts(x_test)
predicted = test_counts.argmax(dim=1)
correct = (predicted == y_test) & (test_counts.max(dim=1).values > 0)

for c, name in enumerate(DIRECTIONS):
    print(f"bar moving {name:5s}: {correct[y_test == c].float().mean():.0%} of {int((y_test == c).sum())} test examples correct")
for every in (1, 2, 3):
    at_speed = torch.tensor(speed_test == every)
    print(f"1 pixel every {every} step{'s' if every > 1 else ' '}: {correct[at_speed].float().mean():.0%} of {int(at_speed.sum())} test examples correct")

# --- the mistakes, animated ---
wrong = torch.nonzero(~correct).flatten()[:4].tolist()  # up to four wrong examples
if wrong:
    examples = [x_test[:, i].reshape(STEPS, SIZE, SIZE).numpy() for i in wrong]
    titles = [f"{DIRECTIONS[y_test[i]]}, 1 px / {speed_test[i]} step{'s' if speed_test[i] > 1 else ''}\nanswer: {DIRECTIONS[predicted[i]] if test_counts[i].max() > 0 else 'none'}" for i in wrong]
    animate(examples, titles, "direction_mistakes.gif", trail=show_trail)

## 5. Export to NIR and run it in py-spinnaker2

As in notebook 03: export to NIR, convert with the rule for snnTorch models. Each test example runs as its own network, so this cell runs only the first 20 test examples on Brian2 (about ten seconds). The answers should match snnTorch; the spike counts can differ by a spike here and there, because of the integer weights.

In [ ]:
# --- export and conversion settings, as in notebook 03 ---
nir_graph = export_to_nir(net, torch.zeros(SIZE * SIZE))
nir_graph.infer_types()  # fill in the input and output shapes of every node

cfg = s2_nir.ConversionConfig()
cfg.dt = 1e-4  # the rule for snnTorch models (notebook 03)
cfg.integrator = s2_nir.IntegratorMethod.FORWARD
cfg.reset = s2_nir.ResetMethod.ZERO
cfg.output_record = ["spikes"]  # record the spikes of the output layer
cfg.conn_delay = 0  # lif_no_delay has no synaptic delays
cfg.scale_weights = True  # scale the float weights into the chip's integer range


def counts_spinnaker2(x_one, backend=brian2):
    """Convert the NIR graph, run one example (time steps, pixels) on a py-spinnaker2 backend, return the output spike counts."""
    s2_net, inputs, outputs = s2_nir.from_nir(nir_graph, cfg)
    inputs[0].params = {pixel: np.nonzero(x_one[:, pixel].numpy())[0].tolist() for pixel in range(SIZE * SIZE)}  # tensor -> spike times
    quiet = isinstance(backend, brian2_sim.Brian2Backend)  # Brian2 prints two progress lines per run; the chip's printout stays
    with contextlib.redirect_stdout(io.StringIO()) if quiet else contextlib.nullcontext():
        backend.run(s2_net, STEPS + 2, record_power=True)  # 2 extra steps for the delay of one step per projection; record_power: chip only
    out = outputs[0].get_spikes()
    return [len(out.get(i, [])) for i in range(4)]


# --- the first 20 test examples on Brian2 ---
same_answer = 0
for i in range(20):
    s2_counts = counts_spinnaker2(x_test[:, i])
    same_answer += int(np.argmax(s2_counts)) == int(predicted[i])
    if i < 5:
        print(f"example {i}: snnTorch {test_counts[i].int().tolist()}, py-spinnaker2 {s2_counts}")
print(f"same answer as snnTorch in {same_answer} of 20 test examples")

## 6. Look inside: what did the network learn?

The hand-built detector was easy to read: its delays said which order of spikes it waits for. Here each hidden neuron has 64 weights, one per pixel. Drawn as an 8 × 8 image, they show which pixels push the neuron up (red) or down (blue). Many show horizontal or vertical stripes: they respond to the orientation of the bar. The direction is harder to see in a still image, because it lies in the timing, in which pixels fire before which. The second cell shows which hidden neurons fire for a clean bar (no noise, no lost events) in each direction.

In [ ]:
# --- one 8 x 8 weight image per hidden neuron ---
w1 = net[0].weight.detach().numpy().reshape(n_hidden, SIZE, SIZE)
limit = np.abs(w1).max()
cols = 8
fig, axes = plt.subplots(int(np.ceil(n_hidden / cols)), cols, figsize=(10, 5))
for k, ax in enumerate(axes.flat):
    ax.set(xticks=[], yticks=[])
    ax.grid(False)
    if k < n_hidden:
        ax.imshow(w1[k], cmap="RdBu_r", vmin=-limit, vmax=limit)
        ax.set_title(f"hidden {k}", fontsize=8)
    else:
        ax.axis("off")
fig.suptitle("weights pixels -> hidden neuron (red: pushes up, blue: pushes down)")
plt.tight_layout()
plt.show()

In [ ]:
# --- a clean bar per direction ---
def clean_bar(direction, every=2, start=2, across=3):
    """A bar without noise or lost events, as input (time steps, 1 example, 64 pixels)."""
    events = np.zeros((STEPS, SIZE, SIZE), dtype=np.float32)
    for k in range(SIZE):
        along = k if direction in ("right", "down") else SIZE - 1 - k
        for i in range(3):
            row, col = (across + i, along) if direction in ("right", "left") else (along, across + i)
            events[start + k * every, row, col] = 1
    return torch.tensor(events.reshape(STEPS, 1, SIZE * SIZE))


# --- hidden spikes per direction ---
hidden_counts = []
for direction in DIRECTIONS:
    utils.reset(net)
    with torch.no_grad():
        hidden = torch.stack([net[1](net[0](x_t)) for x_t in clean_bar(direction)])  # hidden spikes per step
    hidden_counts.append(hidden[:, 0].sum(dim=0).numpy())

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 3))
image = ax.imshow(np.array(hidden_counts), cmap="gray_r", aspect="auto")
ax.set(xlabel="hidden neuron", ylabel="bar moving", xticks=range(n_hidden), yticks=range(4), yticklabels=DIRECTIONS)
ax.tick_params(axis="x", labelsize=7)
ax.grid(False)
fig.colorbar(image, ax=ax, label="spikes")
plt.tight_layout()
plt.show()

Most hidden neurons fire for both directions of one orientation, right and left or down and up: they detect the orientation. A few fire for one direction only. Find one and look at its weight image above: as with the four pixels in notebook 03 ("After the workshop", B), it tends to have red pixels where the bar enters and blue pixels just behind them, so that a bar coming from the other side inhibits it before it can fire.

## 7. Run it on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` and run the cell. It runs the trained network for the first test example on one 48-chip board (about one minute) and compares the answer with snnTorch.

In [ ]:
# --- backend ---
USE_HARDWARE = False  # True: a SpiNNaker2 board (HPI cluster only); False: Brian2, as a dry run

if USE_HARDWARE:
    from spinnaker2 import hardware  # the chip backend, only needed on the cluster

    hw = hardware.SpiNNcloud48NodeBoard()  # one 48-chip board, reserved for this run
else:
    hw = brian2_sim.Brian2Backend()
print("Running on backend:", type(hw).__name__)

# --- run the first test example and compare with snnTorch ---

chip_counts = counts_spinnaker2(x_test[:, 0], backend=hw)
print(f"bar moving {DIRECTIONS[y_test[0]]}: {type(hw).__name__} counts {chip_counts}, snnTorch counts {test_counts[0].int().tolist()}")
print("same answer" if int(np.argmax(chip_counts)) == int(predicted[0]) else "different answer")

## 8. Try it

**Try:** make the leak strong: set `beta = 0.3` in section 2. Predict first: which speed will suffer most? Then run the notebook from section 2 to the end of section 4 (about half a minute) and compare the accuracy per speed with the run before. Set `beta` back to 0.8 afterwards and run the training again.

**If you want more** (after the workshop; the same procedure, one value at a time):

- How small can the hidden layer get? `n_hidden = 8` and `n_hidden = 4`: what happens to the learning curve?
- Fewer training examples: `make_set(200, rng)` for the training set in section 1 and `epochs = 200` in section 2, so that the network gets as many training steps as before. Compare training and test accuracy.
- A harder task: `noise = 0.03` in section 1. Which directions and speeds suffer most?

*Your answer:*

- …

**Discuss:** where did the training happen, and where does the trained network run? What does that mean for using SpiNNaker2 for AI?

*Your answer:*

- …

## 9. Summary

A spiking network that learned to tell the direction of a moving bar on an 8 × 8 sensor, with noise and lost events, at three speeds, and runs on the chip with the same answers. In notebook 02 you designed the detector by hand with delays; here the weights came from examples. Training happens off the chip, on a normal processor; SpiNNaker2 runs the trained network.

**Go deeper:** snnTorch tutorials on training spiking networks: [snntorch.readthedocs.io](https://snntorch.readthedocs.io/en/latest/tutorials/index.html); the NIR workflow on SpiNNaker2: vendor tutorial 08; event cameras and their data: the event-camera demo after this block; workshop notes: [SNNs and deep learning](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning).

---

**End of notebook 04.** Back to the slides: demos and applications.